# 8. Dataclasses

Dataclasses are Python's answer to Java's boilerplate problem. They auto-generate `__init__`, `__repr__`, `__eq__`, and more — just from type-annotated fields. Think of them as **Python's version of Java Records or Lombok's `@Data`**.

This notebook covers: `@dataclass`, default values, `field()`, `frozen`, `__post_init__`, `slots`, ordering, `kw_only`, `asdict()`/`astuple()`, inheritance, and comparison with NamedTuple.

### 8.1 Basic Dataclass

**☕ JAVA (Lombok):**
```java
@Data
public class Employee {
    private String name;
    private String department;
    private double salary;
}
```

**☕ JAVA 16+ (Record):**
```java
public record Employee(String name, String department, double salary) {}
```

**🐍 PYTHON:** `@dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from the annotated fields.

In [ ]:
from dataclasses import dataclass, field, asdict, astuple

@dataclass
class Employee:
    name: str
    department: str
    salary: float
    active: bool = True    # Default value

emp = Employee("Alice", "Engineering", 75000)
print(f"repr: {emp}")                  # Auto __repr__
print(f"name: {emp.name}")              # Direct attribute access
print(f"active: {emp.active}")          # Default value used

# Auto __eq__ — compares by value
emp2 = Employee("Alice", "Engineering", 75000)
print(f"equal: {emp == emp2}")          # True!

### 8.2 The `field()` Function — Advanced Defaults

**☕ JAVA:** No equivalent — mutable defaults aren't a problem in Java constructors.

**🐍 PYTHON:** ⚠️ You **cannot** use a mutable default directly (like `tags: list = []`). Use `field(default_factory=list)` instead.

In [ ]:
@dataclass
class Article:
    title: str
    author: str
    views: int = 0
    tags: list[str] = field(default_factory=list)   # Mutable default!
    _internal_id: str = field(repr=False, default="")  # Hidden from repr

a1 = Article("Python Tips", "Alice")
a1.tags.append("python")

a2 = Article("Java Tips", "Bob")
a2.tags.append("java")

# Each instance has its OWN list (not shared!)
print(f"a1 tags: {a1.tags}")   # ['python']
print(f"a2 tags: {a2.tags}")   # ['java']
print(f"repr: {a1}")           # _internal_id hidden

### 8.3 `__post_init__` — Custom Initialization Logic

**☕ JAVA:** Validation logic goes in the constructor or a `@PostConstruct` method.

**🐍 PYTHON:** `__post_init__` runs **after** the auto-generated `__init__` — perfect for validation or computed fields.

In [ ]:
@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)   # Not a constructor param!

    def __post_init__(self):
        """Runs after __init__ — validate and compute."""
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Dimensions must be positive!")
        self.area = self.width * self.height

rect = Rectangle(10, 5)
print(f"{rect}")   # area is computed automatically

try:
    bad = Rectangle(-1, 5)
except ValueError as e:
    print(f"Validation: {e}")

### 8.4 `frozen=True` — Immutable Dataclasses

**☕ JAVA:** `record` types are immutable by default.

**🐍 PYTHON:** `@dataclass(frozen=True)` makes instances immutable — they can't be modified after creation. This also auto-generates `__hash__`, so they can be used in sets/dicts.

In [ ]:
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(3, 4)
print(f"Point: {p}")

# Can't modify!
try:
    p.x = 10
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

# frozen=True auto-generates __hash__
points = {Point(1, 2), Point(3, 4), Point(1, 2)}
print(f"Unique points: {points}")    # Point(1,2) deduped

### 8.5 `order=True` — Sortable Dataclasses

**☕ JAVA:** Implement `Comparable<T>` with `compareTo()`.

**🐍 PYTHON:** `@dataclass(order=True)` auto-generates `<`, `<=`, `>`, `>=` — comparing **field by field** in declaration order.

In [ ]:
@dataclass(order=True)
class Student:
    gpa: float          # Compared first
    name: str           # Compared second (tiebreaker)

students = [
    Student(3.5, "Charlie"),
    Student(3.9, "Alice"),
    Student(3.5, "Bob"),
]

for s in sorted(students):
    print(f"  {s.gpa} — {s.name}")

print(f"\nHighest GPA: {max(students)}")

### 8.6 `kw_only=True` — Keyword-Only Arguments (Python 3.10+)

**☕ JAVA:** Named parameters don't exist — you rely on IDE hints or the Builder pattern.

**🐍 PYTHON:** `kw_only=True` forces callers to use keyword arguments, preventing positional mistakes. You can also mark individual fields with `field(kw_only=True)`.

In [ ]:
# All fields keyword-only
@dataclass(kw_only=True)
class DatabaseConfig:
    host: str
    port: int = 5432
    database: str = "mydb"
    ssl: bool = True

# Must use keyword arguments — prevents mixing up host/database
config = DatabaseConfig(host="localhost", database="prod", ssl=False)
print(f"{config}")

try:
    bad = DatabaseConfig("localhost")   # ❌ Positional not allowed!
except TypeError as e:
    print(f"❌ {e}")

In [ ]:
# Mix positional and keyword-only fields
@dataclass
class APIRequest:
    url: str                                          # Positional (required)
    method: str = "GET"                               # Positional (optional)
    timeout: int = field(default=30, kw_only=True)    # Keyword-only
    retries: int = field(default=3, kw_only=True)     # Keyword-only

# url and method are positional, timeout and retries must be named
req = APIRequest("https://api.example.com", "POST", timeout=60)
print(f"{req}")

### 8.7 `asdict()` and `astuple()` — Serialization Helpers

**☕ JAVA:** You'd write `toMap()` or use a library like Jackson.

**🐍 PYTHON:** Built-in `asdict()` and `astuple()` convert dataclasses to dicts/tuples — essential for JSON serialization, database insertion, or API responses.

In [ ]:
import json

@dataclass
class Address:
    street: str
    city: str
    country: str = "Greece"

@dataclass
class Person:
    name: str
    age: int
    address: Address   # Nested dataclass!

person = Person("Alice", 30, Address("123 Main St", "Athens"))

# asdict — recursively converts to dict (including nested dataclasses!)
person_dict = asdict(person)
print(f"Dict: {person_dict}")
print(f"JSON: {json.dumps(person_dict, indent=2)}")

# astuple — converts to tuple (flat)
print(f"\nTuple: {astuple(person)}")

> 💡 **Key feature:** `asdict()` handles **nested dataclasses** recursively — the `Address` inside `Person` becomes a nested dict automatically.

### 8.8 Dataclass Inheritance

**☕ JAVA:** `record` types **cannot** be extended — they are implicitly `final`.

**🐍 PYTHON:** Dataclasses support full inheritance! Child fields are appended after parent fields in the constructor.

In [ ]:
@dataclass
class Vehicle:
    make: str
    year: int

@dataclass
class Car(Vehicle):
    doors: int = 4
    fuel: str = "petrol"

@dataclass
class ElectricCar(Car):
    battery_kwh: float = 75.0
    fuel: str = "electric"   # Override parent default!

car = Car("Toyota", 2024)
ev = ElectricCar("Tesla", 2024, doors=4, battery_kwh=100)

print(f"Car: {car}")
print(f"EV:  {ev}")
print(f"\nIs Car: {isinstance(ev, Car)}")        # True
print(f"Is Vehicle: {isinstance(ev, Vehicle)}")  # True

> ⚠️ **Gotcha:** If a parent has fields with defaults, ALL child fields must also have defaults (same Python rule as regular function arguments — no positional after keyword).

### 8.9 `slots=True` — Memory Optimization (Python 3.10+)

**🐍 PYTHON:** `@dataclass(slots=True)` auto-generates `__slots__` — faster attribute access and lower memory usage. No dynamic attribute addition.

In [ ]:
@dataclass(slots=True)
class Coordinate:
    x: float
    y: float
    z: float = 0.0

coord = Coordinate(1.0, 2.0, 3.0)
print(f"Coord: {coord}")

try:
    coord.w = 4.0              # ❌ Can't add attributes with slots!
except AttributeError as e:
    print(f"Blocked: {e}")

### 8.10 Dataclass vs NamedTuple

| Feature | `@dataclass` | `NamedTuple` |
|---------|-------------|-------------|
| Mutable by default | ✅ Yes | ❌ Always immutable |
| Can be frozen | ✅ `frozen=True` | Always frozen |
| Inheritance | ✅ Full class | Limited |
| Is a tuple? | ❌ No | ✅ Yes (supports indexing, unpacking) |
| `__hash__` | Only if frozen or explicit | ✅ Auto |
| Methods | ✅ Full support | ✅ Full support |
| Best for | Most data classes | When you need tuple behavior |

In [ ]:
from typing import NamedTuple

class PointNT(NamedTuple):
    x: float
    y: float

p = PointNT(3, 4)
print(f"NamedTuple: {p}")
print(f"Index:      {p[0]}, {p[1]}")     # Tuple indexing!

x, y = p                                    # Tuple unpacking!
print(f"Unpacked:   x={x}, y={y}")

---

## 🧪 Try It Yourself

**Exercise 1:** Create a `@dataclass` `Product` with `name`, `price`, `quantity` (default 0). Add a computed `total_value` property via `__post_init__`. Make it sortable by price using `order=True`. Use `asdict()` to serialize it to JSON.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `frozen=True` `Color` dataclass with `r`, `g`, `b` (all `int`). Validate in `__post_init__` that values are 0–255. Demonstrate it works in a `set`.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a dataclass hierarchy: `Shape` (base with `color: str`), `Circle(Shape)` with `radius`, and `Rectangle(Shape)` with `width` and `height`. Use `kw_only=True` on `Shape` and add computed `area` via `__post_init__`. Serialize to dict using `asdict()`.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Auto-generated class | Lombok `@Data` / `record` | `@dataclass` |
| Auto `__init__` | Lombok `@AllArgsConstructor` | ✅ Default |
| Auto `__repr__` | Lombok `@ToString` | ✅ Default |
| Auto `__eq__` | Lombok `@EqualsAndHashCode` | ✅ Default |
| Immutable | `record` (Java 16+) | `@dataclass(frozen=True)` |
| Sortable | `Comparable<T>` interface | `@dataclass(order=True)` |
| Keyword-only args | Builder pattern | `@dataclass(kw_only=True)` |
| Mutable default | Not a problem | `field(default_factory=...)` required! |
| Post-construction | `@PostConstruct` | `__post_init__` |
| To dict/tuple | Jackson / manual `toMap()` | `asdict()` / `astuple()` |
| Inheritance | `record` is `final` — can't extend | ✅ Full inheritance support |
| Memory optimization | N/A | `@dataclass(slots=True)` |
| Hidden from repr | Lombok `@ToString.Exclude` | `field(repr=False)` |